# Train a tiny chat model in Colab

This notebook trains a **~14M parameter** chat model from scratch on public
instruction data, then serves it with the repo's FastAPI web demo.

Pipeline:

1. Download Alpaca and convert it to chat JSONL.
2. Pretrain a Base model on the same text so it learns basic English.
3. Supervised fine-tune with assistant-only loss.
4. Print a sample reply and open the browser demo.

**Use a GPU:** Runtime -> Change runtime type -> T4 GPU. On CPU, change
`PROFILE = "tiny"` and lower the step counts.

Quality will be toy-level: short, repetitive answers and no real reasoning. The
point is the complete training and serving path, not ChatGPT.

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/rahullamani01/tiny-llm.git"
if not os.path.isdir("tiny-llm/.git"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir("tiny-llm")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("No GPU detected. Set PROFILE = 'tiny' below or training will be slow.")

## Settings

`small` is the default GPU profile. Use `tiny` for a CPU smoke test and
`medium` only if you have a larger GPU.

In [ ]:
PROFILE = "small"        # "tiny", "small", or "medium"

PROFILES = {
    "tiny":   dict(block=128, embd=192, head=4, layer=4, vocab=2048, batch=8,  grad=1, base_steps=600,  sft_steps=600),
    "small":  dict(block=256, embd=384, head=6, layer=6, vocab=4096, batch=16, grad=2, base_steps=1500, sft_steps=1200),
    "medium": dict(block=256, embd=512, head=8, layer=8, vocab=8192, batch=24, grad=2, base_steps=4000, sft_steps=2500),
}

cfg = PROFILES[PROFILE]
DATA_LIMIT = 8000
DATA_DIR = "data/cache"
CKPT_DIR = "checkpoints/chat"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

def run(*args):
    print("$", " ".join(args), flush=True)
    subprocess.run([sys.executable, *args], check=True)

print(PROFILE, cfg)

## 1. Prepare instruction data

Downloads Alpaca (about 22 MB, cached in `data/cache/`), keeps a shuffled
subset, and writes both chat JSONL and plain text. Change `"alpaca"` to
`"dolly"` in the command below to use Dolly 15k instead.

In [ ]:
run(
    "-m", "data.prepare_chat",
    "--source", "alpaca",
    "--limit", str(DATA_LIMIT),
    "--shuffle",
    "--download-dir", DATA_DIR,
    "--out", f"{DATA_DIR}/chat.jsonl",
    "--text-out", f"{DATA_DIR}/text.txt",
)

print(open(f"{DATA_DIR}/chat.jsonl", encoding="utf-8").readline()[:400])

## 2. Pretrain the Base model

This is ordinary next-token training on the instruction/answer text. It gives
the small model basic fluency before SFT. The BPE vocabulary is learned here and
reused by SFT.

In [ ]:
run(
    "-m", "training.train",
    "--data", f"{DATA_DIR}/text.txt",
    "--out", f"{CKPT_DIR}/base.pt",
    "--tokenizer", "bpe",
    "--vocab-file", f"{CKPT_DIR}/bpe.json",
    "--vocab-size", str(cfg["vocab"]),
    "--bpe-max-chars", "500000",
    "--block-size", str(cfg["block"]),
    "--n-embd", str(cfg["embd"]),
    "--n-head", str(cfg["head"]),
    "--n-layer", str(cfg["layer"]),
    "--batch-size", str(cfg["batch"]),
    "--grad-accum-steps", str(cfg["grad"]),
    "--max-steps", str(cfg["base_steps"]),
    "--eval-interval", "200",
    "--sample-tokens", "0",
    "--device", "auto",
    "--no-plot",
)

## 3. Supervised fine-tuning

Chat special tokens are appended, the embedding/output rows are resized, and
only assistant content plus `<|end|>` contributes to the loss.

In [ ]:
run(
    "-m", "training.sft",
    "--data", f"{DATA_DIR}/chat.jsonl",
    "--init-from", f"{CKPT_DIR}/base.pt",
    "--out", f"{CKPT_DIR}/chat.pt",
    "--block-size", str(cfg["block"]),
    "--n-embd", str(cfg["embd"]),
    "--n-head", str(cfg["head"]),
    "--n-layer", str(cfg["layer"]),
    "--batch-size", str(cfg["batch"]),
    "--grad-accum-steps", str(cfg["grad"]),
    "--max-steps", str(cfg["sft_steps"]),
    "--eval-interval", "200",
    "--device", "auto",
    "--no-plot",
)

## 4. Try a reply

Streams tokens from `chat-best.pt` with the same event path the web server uses.

In [ ]:
from inference.generate import load_model
from server.chat import iter_chat_events

model, tok = load_model(f"{CKPT_DIR}/chat-best.pt")

question = "What is a transformer?"
print("You:", question)
print("Assistant: ", end="")
for event in iter_chat_events(
    model,
    tok,
    [{"role": "user", "content": question}],
    max_new_tokens=80,
    temperature=0.7,
    top_k=40,
):
    if "token" in event:
        print(event["token"], end="", flush=True)
print()

## 5. Open the web demo

Starts the FastAPI server in the background and proxies port 8000 into the
notebook. Stop the cell (or run `server.terminate()`) when you are done.

In [ ]:
import time

from google.colab.output import serve_kernel_port_as_iframe

server = subprocess.Popen([
    sys.executable, "-m", "server.app",
    "--ckpt", f"{CKPT_DIR}/chat-best.pt",
    "--host", "0.0.0.0",
    "--port", "8000",
])

time.sleep(5)
if server.poll() is not None:
    raise RuntimeError("server exited early; check the output above")

serve_kernel_port_as_iframe(8000, height=760)

## 6. Save the checkpoint

Downloads `chat-best.pt` (weights + tokenizer + config). To keep training later,
also download `chat.pt`, which includes optimizer state for `--resume`.

In [ ]:
from google.colab import files

files.download(f"{CKPT_DIR}/chat-best.pt")

## If the runtime disconnects

Re-run the setup cells, then resume SFT from `chat.pt` with the same
architecture and `--resume`. Do not change the model size, tokenizer, or
`--block-size` when resuming.

```bash
python -m training.sft --data data/cache/chat.jsonl \
    --out checkpoints/chat/chat.pt --resume \
    --block-size 256 --n-embd 384 --n-head 6 --n-layer 6
```